In [ ]:
from datasets import load_dataset

# Load dataset
dataset = load_dataset("Zedthecodex/KhmerTimes-Summary")

# Split train/validation
splits = dataset["train"].train_test_split(test_size=0.1, seed=42)
train_dataset = splits["train"]
val_dataset = splits["test"]

print(f"Train size: {len(train_dataset)}")
print(f"Validation size: {len(val_dataset)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/63.0 [00:00<?, ?B/s]

KhmerTimes-Summary-Latest.csv:   0%|          | 0.00/86.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12876 [00:00<?, ? examples/s]

Train size: 11588
Validation size: 1288


In [4]:
!unzip /content/mbart-khmer.zip -d /content/mbart-khmer

Archive:  /content/mbart-khmer.zip
   creating: /content/mbart-khmer/mbart-khmer/
  inflating: /content/mbart-khmer/mbart-khmer/config.json  
  inflating: /content/mbart-khmer/mbart-khmer/generation_config.json  
  inflating: /content/mbart-khmer/mbart-khmer/gitattributes  
  inflating: /content/mbart-khmer/mbart-khmer/model.safetensors  
  inflating: /content/mbart-khmer/mbart-khmer/sentencepiece.bpe.model  
  inflating: /content/mbart-khmer/mbart-khmer/special_tokens_map.json  
  inflating: /content/mbart-khmer/mbart-khmer/tokenizer.json  
  inflating: /content/mbart-khmer/mbart-khmer/tokenizer_config.json  


In [6]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast

model_path = "/content/mbart-khmer/mbart-khmer"
tokenizer = MBart50TokenizerFast.from_pretrained(model_path)
model = MBartForConditionalGeneration.from_pretrained(model_path)


The tokenizer you are loading from '/content/mbart-khmer/mbart-khmer' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [7]:
tokenizer.src_lang = "km_KH"
tokenizer.tgt_lang = "km_KH"

In [17]:
import evaluate

rouge = evaluate.load("rouge")

def evaluate_model(model, tokenizer, dataset, num_samples=100):
    for i in range(min(num_samples, len(dataset))):
        article = dataset[i]["Article"]
        reference = dataset[i]["Summary"]

        inputs = tokenizer(article, return_tensors="pt", truncation=True, max_length=512)
        summary_ids = model.generate(
            **inputs,
            max_length=150,
            num_beams=4,
            forced_bos_token_id=tokenizer.lang_code_to_id["km_KH"]
        )
        generated_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        rouge.add(prediction=generated_summary, reference=reference)

    return rouge.compute()

results = evaluate_model(model, tokenizer, val_dataset, num_samples=50)
print(results)


{'rouge1': np.float64(0.030476190476190476), 'rouge2': np.float64(0.008), 'rougeL': np.float64(0.030476190476190476), 'rougeLsum': np.float64(0.030476190476190476)}
